# 반입 규격을 세우다 — 남의 자료를 들이기 전에

> `notebooks/02-품질·전처리/01.반입규격을세우다.ipynb` · 2026-09-01 · 이동원

**이 노트북이 답하는 것**: *"팀원이 건네준 파일이 쓸 수 있는 것인지 어떻게 아는가."*

지금까지 우리 자료는 **우리가 KRX 에서 직접 받은 것**뿐이었습니다. 모양을 우리가 알고,
틀리면 수집기를 고치면 됐습니다. 이제 팀원이 손으로 모은 파일과 제가 직접 수집한 것이
들어옵니다 — **모양을 우리가 정하지 못합니다.**

그래서 "무엇이 맞는 모양인가"를 먼저 문서로 못박고(**규격**), 그 규격으로 기계가
판정합니다(**해석기**).

이 노트북은 **읽기만 합니다.** DB 를 바꾸지 않습니다.

관련: [반입 규격 README](../../ingest/inbox/schemas/README.md) ·
[기능명세 v1.1](../../docs/기능명세/version1.1/반입_파이프라인.md)

## 0. 준비

In [1]:
import json
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():      # 노트북을 어디서 열든 루트를 찾는다
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# 경로를 세운 뒤에 import 한다 — 노트북을 어디서 열든 저장소 모듈을 찾게 하려면 순서가 이래야 한다
from ingest.inbox.rules import FUNCTIONS, evaluate_rule, referenced_columns  # noqa: E402

SCHEMA_DIR = ROOT / "ingest" / "inbox" / "schemas"
DB = ROOT / "data" / "krx_cache.db"

print("루트    :", ROOT.name)
print("규격    :", ", ".join(sorted(p.stem for p in SCHEMA_DIR.glob("*.json"))))
print("pandas  :", pd.__version__)

루트    : Alpha_Stack
규격    : financial, macro, news, ohlcv_index, ohlcv_stock
pandas  : 3.0.3


## 1. 규격 다섯 장에 무엇이 들어 있나

`fields` 는 어떤 칸을 받는가, `rowRules` 는 어떤 행이 쓸 수 있는가,
`aliases` 는 팀원 파일에서 그 칸이 어떤 이름으로 나타날 수 있는가입니다.

In [2]:
NAMES = ["ohlcv_stock", "ohlcv_index", "news", "financial", "macro"]
specs = {n: json.loads((SCHEMA_DIR / f"{n}.json").read_text(encoding="utf-8")) for n in NAMES}

rows = []
for name, spec in specs.items():
    x = spec["x-alphastack"]
    rows.append({
        "규격": name,
        "제목": spec["title"],
        "필드": len(spec["fields"]),
        "필수": sum(1 for f in spec["fields"] if f.get("constraints", {}).get("required")),
        "규칙": len(x["rowRules"]),
        "error": sum(1 for r in x["rowRules"] if r["severity"] == "error"),
        "별칭": sum(len(v) for v in x["aliases"].values()),
        "대조표": x["target"]["compareWith"] or "—",
    })

pd.DataFrame(rows).set_index("규격")

,제목,필드,필수,규칙,error,별칭,대조표
규격,,,,,,,
ohlcv_stock,종목 시세 (OHLCV),15,3,7,4,85,daily_price
ohlcv_index,지수 시세 (OHLCV),12,2,10,5,82,index_price
news,뉴스 (제목·요약·링크),11,4,11,7,96,—
financial,재무·공시 (DART 재무제표),20,5,10,5,97,—
macro,거시 지표 (ECOS·FRED·KOSIS),13,7,14,10,99,—


## 2. 🔴 가장 많이 틀린 곳 — 널 가드

비교하는 칸이 비어 있을 수 있으면 앞에 `is null` 가드를 답니다.
**널 비교는 참도 거짓도 아닌데 pandas 에서 거짓으로 떨어져 위반이 되기 때문입니다.**

말로만 하면 와닿지 않으니 직접 만들어 봅니다.

In [3]:
보기 = pd.DataFrame({"high": [None, 3.0, 1.0], "low": [1.0, 1.0, 3.0]})

가드없음 = evaluate_rule("high >= low", 보기)
가드있음 = evaluate_rule("high is null or low is null or high >= low", 보기)

pd.DataFrame({
    "high": 보기["high"], "low": 보기["low"],
    "가드 없음": 가드없음, "가드 있음": 가드있음,
})

,high,low,가드 없음,가드 있음
0,NaN,1.0,False,True
1,3.0,1.0,True,True
2,1.0,3.0,False,False


첫 행이 갈립니다. `high` 가 비어 있을 뿐인데 가드가 없으면 **위반**입니다.

이것이 얼마나 큰 문제인지는 실제 자료로 재야 보입니다.

In [4]:
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)      # 읽기 전용으로 연다
지수 = pd.read_sql("SELECT * FROM index_price", con)

빈칸 = 지수[["open", "high", "low", "close"]].isna().sum()
print(f"index_price {len(지수):,}행")
print()
print("빈 칸:")
for k, v in 빈칸.items():
    print(f"  {k:6s} {v:7,}행  ({v / len(지수) * 100:5.2f}%)")

print()
print("가드 없이 `high >= low` 를 error 로 두면:")
막힘 = int((~evaluate_rule("high >= low", 지수)).sum())
print(f"  우리 자신의 자료 {막힘:,}행이 격리된다  ({막힘 / len(지수) * 100:.2f}%)")

index_price 195,864행

빈 칸:
  open    24,933행  (12.73%)
  high    24,933행  (12.73%)
  low     24,933행  (12.73%)
  close    4,097행  ( 2.09%)

가드 없이 `high >= low` 를 error 로 두면:
  우리 자신의 자료 24,933행이 격리된다  (12.73%)


**격리는 "자료가 나쁘다"는 뜻인데, 나쁜 것은 규칙이었습니다.**

같은 결함이 **이미 머지된 종목 규격(PR #28)에도** 있었습니다. 종목 자료는 널이 0건이라
널 문제는 없지만, 다른 이유로 걸립니다 — 거래정지 종목은 시고저가 0 인데 종가는
잔존가로 양수라 `high(0) >= close(양수)` 가 거짓이 됩니다.

In [5]:
종목 = pd.read_sql("SELECT * FROM daily_price", con)
print(f"daily_price {len(종목):,}행")
print()

옛식 = "high >= open and high >= close"
새식 = ("high is null or high == 0 or "
       "((open is null or high >= open) and (close is null or high >= close))")

for 이름, 식 in [("고치기 전", 옛식), ("고친 뒤 ", 새식)]:
    막힘 = int((~evaluate_rule(식, 종목)).sum())
    print(f"  {이름}: {막힘:9,}행 격리  ({막힘 / len(종목) * 100:.2f}%)")

위반 = ~evaluate_rule(옛식, 종목)
zero = (종목.open == 0) & (종목.high == 0) & (종목.low == 0)
겹침 = int((위반 & zero).sum())
그외 = int((위반 & ~zero).sum())
print()
print(f"  고치기 전 격리분 중 zero-OHLC 행 : {겹침:,}")
print(f"  그 밖의 위반                    : {그외:,}   ← 0 이면 원인이 하나뿐이라는 뜻")

daily_price 9,209,812행

  고치기 전:   283,468행 격리  (3.08%)
  고친 뒤 :         0행 격리  (0.00%)

  고치기 전 격리분 중 zero-OHLC 행 : 283,468
  그 밖의 위반                    : 0   ← 0 이면 원인이 하나뿐이라는 뜻


## 3. 규칙이 우리 자료를 격리하지 않는지 전부 확인

`error` 규칙이 우리 자신의 자료를 한 행이라도 격리하면 **규칙이 틀린 것**입니다.
팀원 파일을 받기 전에 여기서 먼저 압니다.

In [6]:
def 검사(규격이름: str, 표이름: str) -> pd.DataFrame:
    spec = specs[규격이름]
    frame = pd.read_sql(f"SELECT * FROM {표이름}", con)
    out = []
    for rule in spec["x-alphastack"]["rowRules"]:
        어김 = int((~evaluate_rule(rule["expr"], frame)).sum())
        out.append({
            "규칙": rule["id"],
            "심각도": rule["severity"],
            "위반": 어김,
            "비율%": round(어김 / len(frame) * 100, 3),
        })
    return pd.DataFrame(out)

결과_종목 = 검사("ohlcv_stock", "daily_price")
결과_지수 = 검사("ohlcv_index", "index_price")

print("=== ohlcv_stock ← daily_price (9,209,812행) ===")
display(결과_종목)
print("=== ohlcv_index ← index_price (195,864행) ===")
display(결과_지수)

=== ohlcv_stock ← daily_price (9,209,812행) ===


,규칙,심각도,위반,비율%
0,high_ge_low,error,0,0.000
1,high_ge_open_close,error,0,0.000
2,low_le_open_close,error,0,0.000
3,zero_ohlc,warn,283468,3.078
4,zero_ohlc_but_traded,warn,125,0.001
5,not_future,error,0,0.000
6,is_business_day,warn,0,0.000


=== ohlcv_index ← index_price (195,864행) ===


,규칙,심각도,위반,비율%
0,high_ge_low,error,0,0.000
1,high_ge_open_close,error,0,0.000
2,low_le_open_close,error,0,0.000
3,ohl_missing_together,warn,0,0.000
4,ohl_zero_not_missing,warn,0,0.000
5,has_close_or_volume,error,0,0.000
6,close_present,warn,4097,2.092
7,change_rate_matches_change,warn,0,0.000
8,not_future,error,0,0.000
9,is_business_day,warn,0,0.000


In [7]:
for 이름, 결과 in [("종목", 결과_종목), ("지수", 결과_지수)]:
    막힘 = 결과.loc[결과["심각도"] == "error", "위반"].sum()
    표시 = 결과.loc[결과["심각도"] == "warn", "위반"].sum()
    상태 = "✅ 한 행도 격리하지 않는다" if 막힘 == 0 else f"❌ {막힘:,}행 격리 — 규칙이 틀렸다"
    print(f"{이름}: {상태}   (warn 으로 표시만 하는 행 {표시:,})")

종목: ✅ 한 행도 격리하지 않는다   (warn 으로 표시만 하는 행 283,593)
지수: ✅ 한 행도 격리하지 않는다   (warn 으로 표시만 하는 행 4,097)


`warn` 건수가 세션 기록의 실측값과 맞는지도 봅니다. **통과만 보고는 규칙이
뜻대로 읽혔는지 알 수 없습니다 — 값이 맞아야 압니다.**

In [8]:
# 세션 기록에 남아 있던 실측값. 규칙이 이 숫자를 그대로 세야 뜻대로 읽힌 것이다.
알고있던값 = {"zero_ohlc": 283_468, "zero_ohlc_but_traded": 125}
잰값 = 결과_종목.set_index("규칙")["위반"].to_dict()

for 규칙, 기대 in 알고있던값.items():
    실제 = 잰값[규칙]
    맞나 = "✅ 일치" if 실제 == 기대 else f"❌ 어긋남 (기대 {기대:,})"
    print(f"  {규칙:24s} 실측 {실제:9,}   {맞나}")

  zero_ohlc                실측   283,468   ✅ 일치
  zero_ohlc_but_traded     실측       125   ✅ 일치


## 4. 해석기는 어떻게 읽는가

`expr` 은 통째로 **유효한 파이썬 식**입니다. 그래서 문법을 새로 만들지 않고
`ast.parse` 로 읽습니다 — 우선순위는 파이썬 파서가 이미 옳게 알고 있으니 빌립니다.

⚠️ 처음에는 `and` → `&` 문자열 치환으로 옮겼습니다. 그런데 파이썬에서 비트 연산자가
비교보다 **먼저** 묶여 `a | b == 0` 이 `a | (b == 0)` 이 됩니다.

In [9]:
# 문자열 치환이 만들어 낸 것과 ast 로 읽은 것을 나란히 놓는다
치환식 = "df['high'].isna() | df['high'] == 0 | (df['high'] >= df['low'])"
바른식 = "high is null or high == 0 or high >= low"

df = 지수.head(200000)
try:
    치환결과 = eval(치환식, {"df": df})       # noqa: S307 — 무엇이 틀리는지 보이려고 일부러 실행
    print("치환식 위반:", int((~치환결과.fillna(False)).sum()), "행")
except Exception as e:
    print("치환식은 아예 터진다:", type(e).__name__, str(e)[:80])

print("바른식 위반:", int((~evaluate_rule(바른식, df)).sum()), "행")
print()
print(f"허용하는 함수 {len(FUNCTIONS)}개: {', '.join(sorted(FUNCTIONS))}")
print("참조하는 칸:", sorted(referenced_columns(바른식)))

치환식 위반: 27421 행
바른식 위반: 0 행

허용하는 함수 10개: abs, contains, date, day, len, matches, month, starts_with, time, year
참조하는 칸: ['high', 'low']


## 5. 미래참조 — 도메인마다 기준이 다르다

시세는 하루 단위라 "거래일 T 는 T+1 0시부터"로 끝납니다. 나머지는 다릅니다.

In [10]:
표 = []
for name, spec in specs.items():
    la = spec["x-alphastack"]["lookahead"]
    if la is None:
        표.append({"규격": name, "시점 기준": "—", "요약": "lookahead 없음"})
        continue
    기준 = la.get("knownAtField") or la.get("timeField") or la.get("field") or "bas_dd"
    요약 = (la.get("knownAtRule") or la.get("effectiveFrom") or la.get("rule") or "")
    표.append({"규격": name, "시점 기준": 기준, "요약": 요약[:70]})

pd.DataFrame(표).set_index("규격")

,시점 기준,요약
규격,,
ohlcv_stock,—,lookahead 없음
ohlcv_index,bas_dd,bas_dd + 1일 00:00 (KST)
news,pub_dt,뉴스의 known_at 은 발행 시각 그 자체다. 수집 시각(collected_at...
financial,rcept_dt,next_business_day(rcept_dt)
macro,known_from,release_date 가 있고 release_after_period_end·rel...


### 거래일은 계산으로 못 정합니다

`common.trading_calendar.trading_days()` 는 **주말만** 건너뛰고 공휴일을 모릅니다.
뉴스 배정("다음 거래일")과 재무("접수일 다음 거래일")가 이것에 기대면 어긋납니다.

In [11]:
from datetime import date, timedelta

거래일 = [r[0] for r in con.execute(
    "SELECT DISTINCT bas_dd FROM daily_price WHERE bas_dd <= '20210831' ORDER BY bas_dd")]

시작 = date(int(거래일[0][:4]), int(거래일[0][4:6]), int(거래일[0][6:]))
끝 = date(int(거래일[-1][:4]), int(거래일[-1][4:6]), int(거래일[-1][6:]))

평일 = 0
d = 시작
while d <= 끝:
    if d.weekday() < 5:
        평일 += 1
    d += timedelta(days=1)

print(f"개발구간  {거래일[0]} ~ {거래일[-1]}")
print(f"  실제 거래일 {len(거래일):,}일")
print(f"  평일        {평일:,}일")
print(f"  차이        {평일 - len(거래일):,}일  ← 공휴일. 주말 규칙만 쓰면 "
      f"{(평일 - len(거래일)) / 평일 * 100:.1f}% 틀린다")

개발구간  20100104 ~ 20210831
  실제 거래일 2,880일
  평일        3,042일
  차이        162일  ← 공휴일. 주말 규칙만 쓰면 5.3% 틀린다


## 6. 정리

| 무엇 | 결과 |
|---|---|
| 규격 | 5장 (`ohlcv_stock`·`ohlcv_index`·`news`·`financial`·`macro`) |
| `error` 규칙이 우리 자료를 격리하는가 | **하지 않는다** (종목·지수 둘 다 0행) |
| 이미 나간 규격에서 찾은 결함 | 종목 `high_ge_open_close` 가 283,468행(3.1%) 격리 → 고침 |
| 해석기 | `ast` 로 읽는다. `eval` 을 쓰지 않는다 |
| 거래일 | 계산 불가 — `daily_price.bas_dd` 실측 집합을 쓴다 |

**다음**: `scripts/check_inbox.py`(세션마다 새 파일 확인 + 자동 적재)와
`inbox_accepted`·`inbox_quarantine` 표를 만듭니다.

In [12]:
con.close()
print("읽기 전용 연결을 닫았습니다. DB 는 바뀌지 않았습니다.")

읽기 전용 연결을 닫았습니다. DB 는 바뀌지 않았습니다.
